# BioRAG-Hybrid: GPU embedding generation in Google Colab

This notebook generates frozen BioBERT and MedCPT document and question embeddings. It does not train models. At the end, it downloads one ZIP containing the four NPZ caches and their metadata sidecars.

Before running: in Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Upload `artifacts/colab/biorag_colab_bundle.zip` when prompted. After download, extract the output ZIP into this repository's `data/official/embeddings/` folder and follow `data/official/embeddings/README.md` to run evaluation locally.

The initial bundle is a compact subset of the project: the official question file, PubMed corpus, loader, and embedding script. No Google Drive mount or API key is required. Upload/download still transfers the files through your browser.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab select Runtime → Change runtime type → GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()  # Select artifacts/colab/biorag_colab_bundle.zip
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Upload exactly one project bundle ZIP; received: {list(uploaded)}')
BUNDLE_DIR = Path('/content/biorag_project')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(BUNDLE_DIR)
matches = list(BUNDLE_DIR.rglob('tools/encode_embeddings.py'))
if len(matches) != 1:
    raise FileNotFoundError('Bundle did not contain exactly one tools/encode_embeddings.py')
PROJECT_ROOT = matches[0].parents[1]
print('Project bundle:', PROJECT_ROOT)
print('Question file:', PROJECT_ROOT / 'data/official/training11b.json')
print('PubMed corpus:', PROJECT_ROOT / 'data/official/pubmed_abstracts.json')

In [ ]:
# Keep Colab's preinstalled PyTorch/CUDA build. Install the encoder and cache dependencies.
%pip -q install 'transformers>=4.40,<5' 'numpy>=1.24,<3'

In [ ]:
from pathlib import Path
import subprocess

DATA_DIR = PROJECT_ROOT / 'data/official'
OUTPUT_DIR = Path('/content/biorag_embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 8  # Lower this to 4 or 2 if the GPU runs out of memory.

def encode(model, kind, input_path, output_name, max_length):
    command = [
        'python', str(PROJECT_ROOT / 'tools/encode_embeddings.py'),
        '--kind', kind, '--model', model,
        '--input', str(input_path), '--output', str(OUTPUT_DIR / output_name),
        '--batch-size', str(BATCH_SIZE), '--max-length', str(max_length), '--device', 'cuda',
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

print('Output directory:', OUTPUT_DIR)

## Generate BioBERT caches

BioBERT is used here as a frozen embedding baseline with masked mean pooling. The document vectors combine each title and abstract; the default maximum length is 512 tokens for documents and 64 for questions. This cell may take a while on the full corpus.

In [ ]:
encode('biobert', 'documents', DATA_DIR / 'pubmed_abstracts.json', 'biobert_documents.npz', 512)
encode('biobert', 'questions', DATA_DIR / 'training11b.json', 'biobert_queries.npz', 64)

## Generate MedCPT caches

MedCPT uses its separate pretrained query and article encoders, with `[CLS]` vectors. The article encoder receives title/abstract pairs. This cell is independent of the BioBERT cell, so completed BioBERT caches remain available if you need to adjust the MedCPT batch size.

In [ ]:
encode('medcpt-article', 'documents', DATA_DIR / 'pubmed_abstracts.json', 'medcpt_documents.npz', 512)
encode('medcpt-query', 'questions', DATA_DIR / 'training11b.json', 'medcpt_queries.npz', 64)

## Package and download the caches

This checks that all four vector files and their provenance sidecars exist, creates one ZIP, and starts a browser download. Keep the `.npz.json` sidecars with the corresponding `.npz` files.

In [ ]:
from google.colab import files
import zipfile

cache_names = [
    'biobert_documents.npz', 'biobert_documents.npz.json',
    'biobert_queries.npz', 'biobert_queries.npz.json',
    'medcpt_documents.npz', 'medcpt_documents.npz.json',
    'medcpt_queries.npz', 'medcpt_queries.npz.json',
]
missing = [name for name in cache_names if not (OUTPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing cache outputs: {missing}')
CACHE_ZIP = Path('/content/biorag_embedding_caches.zip')
with zipfile.ZipFile(CACHE_ZIP, 'w', compression=zipfile.ZIP_STORED) as archive:
    for name in cache_names:
        archive.write(OUTPUT_DIR / name, arcname=name)
print('Cache archive size (MB):', round(CACHE_ZIP.stat().st_size / (1024 * 1024), 1))
files.download(str(CACHE_ZIP))

## Run the local evaluator after download

Extract `biorag_embedding_caches.zip` into `data/official/embeddings/` in the local project. Then run the CPU evaluator from the repository root:

```bash
python tools/evaluate_retrieval.py \
  --biobert-docs data/official/embeddings/biobert_documents.npz \
  --biobert-queries data/official/embeddings/biobert_queries.npz \
  --medcpt-docs data/official/embeddings/medcpt_documents.npz \
  --medcpt-queries data/official/embeddings/medcpt_queries.npz \
  --output artifacts/results/neural_retrieval_evaluation.json
```

The evaluator selects the hybrid weights on validation questions before calculating the held-out test metrics.